# Okaí: 1. Limpeza da base de hotéis

**Entrada:** `hoteis_cnae_5510801.csv`, a base de CNPJ passada pelo professor.

**Saída:** `hoteis.csv` (lista para o site) e `cidades.json` (resumo por cidade).

**Antes de rodar:** clique no ícone de pasta, à esquerda do Colab, e arraste o CSV para lá. Depois use *Ambiente de execução > Executar tudo*.

In [ ]:
import json
import pandas as pd

ARQUIVO = "hoteis_cnae_5510801.csv"
base = pd.read_csv(ARQUIVO, dtype=str, low_memory=False)
print(f"Registros na base original: {len(base):,}".replace(",", "."))
base.head(3)

## Filtros
Cada filtro abaixo mostra quantos hotéis restam, para que vocês possam explicar as escolhas na apresentação.

- **Situação 02:** mantém só empresas ativas.
- **Brasil:** remove os registros marcados como exterior (`EX`).
- **Nome fantasia e município:** são necessários para exibir o hotel e ligá-lo à cidade.
- **Palavra de hospedagem no nome:** ajuda a retirar nomes como "ML Serviços" ou nomes de pessoas. Pode retirar também hotéis reais com nomes diferentes, o que é uma limitação a citar.

In [ ]:
PALAVRAS = (r"\b(?:HOTEL|HOTEIS|HOTÉIS|POUSADA|POUSADAS|HOSTEL|HOSTELS|RESORT|INN|LODGE|FLAT|FLATS|"
            r"HOSPEDARIA|ALBERGUE|CHALE|CHALES|CHALÉ|CHALÉS|SUITES|APART|PENSAO|PENSÃO|ECOLODGE|GUEST)\b")

passos = []
df = base.copy()
passos.append(("Base original", len(df)))

df = df[df["situacao_cadastral"] == "02"]
passos.append(("Empresas ativas", len(df)))

df = df[df["uf"] != "EX"]
passos.append(("No Brasil", len(df)))

df = df[df["nome_fantasia"].notna() & df["codigo_ibge"].notna() & df["municipio"].notna()]
passos.append(("Com nome e município", len(df)))

df = df[df["nome_fantasia"].str.contains(PALAVRAS, case=False, regex=True)]
passos.append(("Com palavra de hospedagem", len(df)))

pd.DataFrame(passos, columns=["Etapa", "Hotéis"])

## Colunas do site
Mantemos só o que ajuda a escolher um hotel. **CNPJ, razão social, telefone e e-mail ficam de fora**, e o número do endereço também.

In [ ]:
MINUSCULAS = {"da", "de", "do", "das", "dos", "e"}

def formatar(nome):
    if not isinstance(nome, str):
        return ""
    palavras = nome.strip().lower().split()
    return " ".join(p if p in MINUSCULAS and i > 0 else p.capitalize()
                    for i, p in enumerate(palavras))

def tipo(nome):
    n = nome.upper()
    if "POUSADA" in n: return "Pousada"
    if "HOSTEL" in n or "ALBERGUE" in n: return "Hostel"
    if "RESORT" in n: return "Resort"
    return "Hotel"

hoteis = pd.DataFrame({
    "nome": df["nome_fantasia"].map(formatar),
    "tipo": df["nome_fantasia"].map(tipo),
    "bairro": df["bairro"].map(formatar),
    "cidade": df["municipio"],
    "uf": df["uf"],
    "codigo_ibge": df["codigo_ibge"],
    "ano_inicio": df["data_inicio_atividades"].str[:4],
})

antes = len(hoteis)
hoteis = hoteis.drop_duplicates(subset=["nome", "codigo_ibge"]).sort_values(["uf", "cidade", "nome"])
print(f"Nomes repetidos na mesma cidade removidos: {antes - len(hoteis)}")
hoteis.sample(5, random_state=1)

## Resumo por cidade
É este arquivo que vai alimentar o mapa e a busca.

In [ ]:
cidades = (hoteis.groupby(["codigo_ibge", "cidade", "uf"])
           .agg(total=("nome", "size"),
                hoteis=("tipo", lambda s: int((s == "Hotel").sum())),
                pousadas=("tipo", lambda s: int((s == "Pousada").sum())),
                hostels=("tipo", lambda s: int((s == "Hostel").sum())),
                resorts=("tipo", lambda s: int((s == "Resort").sum())))
           .reset_index()
           .sort_values("total", ascending=False))

print(f"{len(hoteis):,} hospedagens em {len(cidades):,} cidades".replace(",", "."))
cidades.head(10)

## Salvar
Os dois arquivos aparecem na pasta à esquerda do Colab e são baixados automaticamente.

In [ ]:
hoteis.to_csv("hoteis.csv", index=False, encoding="utf-8-sig")
with open("cidades.json", "w", encoding="utf-8") as f:
    json.dump(cidades.to_dict(orient="records"), f, ensure_ascii=False)

try:
    from google.colab import files
    files.download("hoteis.csv")
    files.download("cidades.json")
except ImportError:
    print("Fora do Colab: arquivos salvos na pasta atual.")